# Breast Cancer Classification using SVM

In this section, we apply Support Vector Machines to a real-world medical dataset:

**Breast Cancer Wisconsin Dataset**

The goal is to classify tumors as:

- Malignant (cancerous)
- Benign (non-cancerous)

This is a binary classification problem widely used to evaluate machine learning models in healthcare.

##  Import Libraries and Load Dataset

We use the built-in dataset from Scikit-Learn.

In [1]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score
)

In [2]:
data = load_breast_cancer()

X = data.data
y = data.target

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

print("Training Samples:", X_train.shape[0])
print("Testing Samples:", X_test.shape[0])

Training Samples: 426
Testing Samples: 143


### Feature Scaling (IMPORTANT)

SVM is sensitive to feature magnitude.

We apply StandardScaler to normalize the dataset before training models.

In [3]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

## Kernel Comparison

Support Vector Machines can use different kernel functions to transform data into higher-dimensional feature spaces.

In this experiment, we compare:

**Linear Kernel**

Decision boundary:

w · x + b = 0

The linear kernel works well when data is linearly separable.

**Polynomial Kernel**

Kernel function:

K(x,y) = (x · y + c)^d

where:

- c is a constant
- d is the polynomial degree

**Radial Basis Function (RBF) Kernel**

Kernel function:

K(x,y) = exp(-γ ||x-y||²)

where:

- γ controls the influence of training samples
- Larger γ creates more complex boundaries

In [4]:
kernels={
    "Linear": SVC(kernel="linear",C=1.0),
    "Poly": SVC(kernel="poly",degree=3,C=1.0),
    "RBF": SVC(kernel="rbf",gamma=0.01,C=1.0),
}

models = {}

for name, model in kernels.items():
    model.fit(X_train, y_train)
    models[name] = model

def evaluate_model(name, model):
    pred = model.predict(X_test)

    report = {
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "F1 Score": f1_score(y_test, pred),
        "ROC AUC": roc_auc_score(y_test, pred),
        "Confusion Matrix": confusion_matrix(y_test, pred)
    }

    return report

results = []

for name, model in models.items():
    results.append(evaluate_model(name, model))

for r in results:
    print("\n==============================")
    print("Model:", r["Model"])
    print("Accuracy:", round(r["Accuracy"], 4))
    print("F1 Score:", round(r["F1 Score"], 4))
    print("ROC AUC:", round(r["ROC AUC"], 4))
    print("Confusion Matrix:\n", r["Confusion Matrix"])


Model: Linear
Accuracy: 0.986
F1 Score: 0.9889
ROC AUC: 0.985
Confusion Matrix:
 [[52  1]
 [ 1 89]]

Model: Poly
Accuracy: 0.9021
F1 Score: 0.9278
ROC AUC: 0.8679
Confusion Matrix:
 [[39 14]
 [ 0 90]]

Model: RBF
Accuracy: 0.972
F1 Score: 0.978
ROC AUC: 0.9661
Confusion Matrix:
 [[50  3]
 [ 1 89]]


## Hyperparameter Tuning (C Parameter)

We tune the regularization parameter **C** for the RBF kernel.

C controls the trade-off between:

- Maximizing margin
- Minimizing classification error

In [5]:
best_acc=0
best_C=None

for C in [0.1,1,10,50]:
    svm=SVC(kernel="rbf",C=C,gamma=0.01)
    svm.fit(X_train,y_train)
    y_pred=svm.predict(X_test)
    acc=accuracy_score(y_test,y_pred)

    print(f"C={C:>4}: Accuracy={acc:.4f}")

    if acc>best_acc:
        best_acc=acc
        best_C=C

print(f"Best C selected: {best_C}")

C= 0.1: Accuracy=0.9441
C=   1: Accuracy=0.9720
C=  10: Accuracy=0.9790
C=  50: Accuracy=0.9790
Best C selected: 10
